## Analysing 250 ranked Lulu games on KR

This notebook walks through the ETL pipeline behind the [league-analysis](https://github.com/RubenPeeters/league-analysis) project.
We pull match data from the Riot Games API, normalise it into DataFrames, and answer one question:
**what should I build and run against each lane opponent?**

### 1 — Data collection

The API client fetches match history, match details, and timelines, caching everything to disk so re-runs are free. We pull 250 ranked solo-queue games (queue 420) and resolve item/rune names from Data Dragon.

In [1]:
from api import RiotAPIClient
from transform import build_dataframes

client = RiotAPIClient(api_key)
puuid = client.get_puuid(game_name, tag_line)
match_ids = client.get_match_ids(puuid, count=250, queue=420)

matches, timelines = {}, {}
for mid in match_ids:
    matches[mid] = client.get_match(mid)
    timelines[mid] = client.get_timeline(mid)

print(f"Loaded {len(match_ids)} ranked matches")
print(f"Patches encountered: {', '.join(sorted(set(patches)))}")
print(f"Item names resolved: {len(item_names)} | Rune names resolved: {len(rune_names)}")

Loaded 250 ranked matches
Patches encountered: 15.24, 16.1, 16.2, 16.3
Item names resolved: 278 | Rune names resolved: 63


### 2 — Building the DataFrames

The `build_dataframes` function filters to Lulu games, identifies the lane opponent, extracts starting/core items via a timeline state-machine, and parses rune pages. The result is three normalised tables.

In [2]:
matches_df, items_df, runes_df = build_dataframes(
    matches, timelines, puuid, champion_name="Lulu"
)

print(f"Lulu games with a lane opponent: {len(matches_df)}")
print(f"Item records: {len(items_df):,} | Rune records: {len(runes_df):,}")

Lulu games with a lane opponent: 164
Item records: 1,147 | Rune records: 984


In [3]:
matches_df.head()

,Match_ID,Enemy_Champion,Win
0,KR_7954210624,Sion,True
1,KR_7954154772,Rumble,True
2,KR_7954078227,Aurora,False
3,KR_7952658522,Ambessa,True
4,KR_7952651201,Jax,True


### 3 — Win rate per matchup

Group by opponent and look at sample size + win rate. We filter to matchups with at least 3 games.

In [4]:
import pandas as pd

summary = (
    matches_df.groupby("Enemy_Champion")
    .agg(Games=("Win", "count"), Wins=("Win", "sum"))
    .query("Games >= 3")
    .sort_values("Games", ascending=False)
)
summary["Win Rate"] = (summary["Wins"] / summary["Games"] * 100).round().astype(int).astype(str) + "%"
summary.index.name = "Enemy"
summary.head(12)

,Games,Wins,Win Rate
Enemy,,,
Sion,17,11,65%
Rumble,13,9,69%
Zaahen,13,9,69%
Aurora,11,8,73%
Ambessa,10,5,50%
Jax,10,8,80%
Renekton,10,7,70%
Ryze,9,5,56%
Irelia,7,3,43%


### 4 — Keystone distribution

Electrocute dominates across most matchups. Phase Rush and Summon Aery appear situationally.

In [5]:
keystones = runes_df[runes_df["Is_Keystone"]].merge(
    matches_df[["Match_ID"]], on="Match_ID"
)
counts = keystones["Rune_ID"].value_counts()

print("Keystone usage across all Lulu games:")
for rune_id, cnt in counts.items():
    name = rune_names.get(rune_id, f"Unknown({rune_id})")
    print(f"  {name:<18} {cnt:>3}  ({cnt / len(matches_df) * 100:>2.0f}%)")

Keystone usage across all Lulu games:
  Electrocute      138  (84%)
  Summon Aery       11  ( 7%)
  Arcane Comet       8  ( 5%)
  Phase Rush         5  ( 3%)
  Press the Attack   4  ( 2%)


### 5 — Best and worst matchups

Looking at matchups with 5+ games to reduce noise.

In [6]:
qualified = summary.query("Games >= 5").copy()
qualified["WR"] = qualified["Wins"] / qualified["Games"]

print("Best matchups (5+ games):")
for enemy, row in qualified.nlargest(5, "WR").iterrows():
    print(f"  {enemy:<13} {row['Games']:>2} games  {row['WR']:>4.0%} WR")

print("\nWorst matchups (5+ games):")
for enemy, row in qualified.nsmallest(5, "WR").iterrows():
    print(f"  {enemy:<13} {row['Games']:>2} games  {row['WR']:>4.0%} WR")

Best matchups (5+ games):
  Fiora         5 games  100% WR
  Vladimir      6 games   83% WR
  Jax          10 games   80% WR
  Aurora       11 games   73% WR
  Renekton     10 games   70% WR

Worst matchups (5+ games):
  K'Sante       7 games   29% WR
  Irelia        7 games   43% WR
  Ambessa      10 games   50% WR
  Ryze          9 games   56% WR
  Yone          5 games   60% WR


### 6 — Takeaways

- **Electrocute** is the default keystone (~84% of games). Phase Rush comes out against Darius and Rumble.
- **Doran's Ring + 2× Health Potion** is the universal starting buy.
- The toughest matchups are K'Sante (29%) and Irelia (43%) — both have gap closers that punish Lulu's short range.
- Lulu top crushes immobile melee: Fiora (100%), Vladimir (83%), Jax (80%).
- Build adapts per matchup: Plated Steelcaps into AD threats, Mercury's Treads into AP, Boots of Swiftness into tanks.